# dispel4py multi-worker agent workflow — simplified, with complete traces

This notebook contains **only the parallel/multi-worker use case**. It is designed to make three things obvious:

1. which events the deterministic precheck resolves without an agent;
2. which suspicious events are routed to an LLM-agent worker; and
3. exactly what each agent did, including every tool call and tool result.

The default dataset contains **60 events**:

| Route | Cases | Events | Agent needed? |
|---|---|---:|---|
| Clearly normal | routine and stable | 20 | No |
| Battery precheck | battery below 10% | 5 | No |
| Temperature precheck | outside -40°C to 85°C | 5 | No |
| Suspicious | six kinds of ambiguous evidence | 30 | Yes |

Therefore, **50% of the events reach an agent**. Change `SCENARIO_COUNTS` if you want a cheaper or larger run.

> This is a parallel multi-worker workflow, not a system in which agents talk to each other. Four independent instances of the same Agent PE process different events. An event is handled by one agent worker. Previous and neighbouring readings are snapshots embedded in that event before execution.

## 1. Install dispel4py and dependencies

In [ ]:
%cd /content
!rm -rf /content/d4py
!git clone -q https://github.com/StreamingFlow/d4py.git /content/d4py
%pip uninstall -y -q stream-d4py stream_d4py dispel4py
%pip install -qU pip setuptools wheel
%pip install -q mpi4py openai pandas matplotlib
%pip install -q -e /content/d4py

from pathlib import Path
Path("/content/results").mkdir(exist_ok=True)
print("Installation complete.")

## 2. Add the API key

The key is entered without being displayed and remains in this Colab runtime. Thirty events reach an agent in the default run, and an event may require more than one model turn, so the run can incur API charges.

In [ ]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key: ")

os.environ.setdefault("OPENAI_MODEL", "gpt-5-mini")
print("Model:", os.environ["OPENAI_MODEL"])

## 3. Create a deliberately balanced dataset

The generator uses named scenarios instead of hiding anomalies at a few event numbers. This makes the intended route inspectable before the workflow runs.

The six agent cases are:

- `packet_loss`: communication may be temporary; inspect history and consider a retry;
- `calibration_drift`: compare history/neighbours and consider maintenance;
- `moisture_buzzing`: possible enclosure/hardware problem;
- `isolated_temperature_spike`: determine whether the sensor alone changed;
- `neighbour_disagreement`: reading conflicts with nearby sensors;
- `intermittent_fault`: vague but potentially important hardware warning.

`previous_readings` and `neighbour_readings` are attached snapshots. Calling the neighbour tool does **not** contact another agent.

In [ ]:
import json
import random
from collections import Counter, defaultdict, deque
from datetime import datetime, timedelta, timezone

random.seed(42)

SCENARIO_COUNTS = {
    "clear_normal": 20,
    "low_battery": 5,
    "impossible_temperature": 5,
    "packet_loss": 5,
    "calibration_drift": 5,
    "moisture_buzzing": 5,
    "isolated_temperature_spike": 5,
    "neighbour_disagreement": 5,
    "intermittent_fault": 5,
}

AGENT_SCENARIOS = {
    "packet_loss",
    "calibration_drift",
    "moisture_buzzing",
    "isolated_temperature_spike",
    "neighbour_disagreement",
    "intermittent_fault",
}

scenario_plan = [
    scenario
    for scenario, count in SCENARIO_COUNTS.items()
    for _ in range(count)
]
random.shuffle(scenario_plan)

sensor_ids = [f"sensor-{i:03d}" for i in range(1, 11)]
neighbour_map = {
    sensor: [
        other for j, other in enumerate(sensor_ids)
        if abs(i - j) == 1
    ]
    for i, sensor in enumerate(sensor_ids)
}

history = defaultdict(lambda: deque(maxlen=5))
latest = {}
events = []
start = datetime(2026, 8, 6, 9, 0, tzinfo=timezone.utc)

for number, scenario in enumerate(scenario_plan):
    sensor = sensor_ids[number % len(sensor_ids)]
    base_temperature = 20 + (number % len(sensor_ids)) * 0.35
    temperature = round(base_temperature + random.gauss(0, 0.2), 2)
    humidity = round(45 + random.gauss(0, 1.5), 2)
    battery = 80 - number // 5
    note = "Routine reading. Signal is stable."

    if scenario == "low_battery":
        battery = 5
        note = "Reading is plausible, but the battery is critically low."
    elif scenario == "impossible_temperature":
        temperature = 96.0
        note = "Temperature exceeds the sensor's plausible operating range."
    elif scenario == "packet_loss":
        note = "Several packets were lost during transmission; the connection returned."
    elif scenario == "calibration_drift":
        temperature = round(temperature + 7.0, 2)
        note = "Possible calibration drift after a gradual offset was observed."
    elif scenario == "moisture_buzzing":
        humidity = 88.0
        note = "Intermittent buzzing; moisture may have entered the enclosure."
    elif scenario == "isolated_temperature_spike":
        temperature = round(temperature + 15.0, 2)
        note = "Sudden isolated temperature increase with no confirmed environmental cause."
    elif scenario == "neighbour_disagreement":
        temperature = round(temperature + 11.0, 2)
        note = "Reading disagrees with nearby sensors; cause is uncertain."
    elif scenario == "intermittent_fault":
        note = "Intermittent fault indicator appeared and then cleared without explanation."

    event = {
        "event_id": f"event-{number:04d}",
        "sensor_id": sensor,
        "zone": "north",
        "timestamp": (start + timedelta(minutes=number)).isoformat(),
        "temperature": temperature,
        "humidity": humidity,
        "battery": battery,
        "diagnostic_note": note,
        "scenario_type": scenario,
        "expected_route": "agent" if scenario in AGENT_SCENARIOS else "deterministic",
        "previous_readings": list(history[sensor]),
        "neighbour_readings": [
            latest[n] for n in neighbour_map[sensor] if n in latest
        ],
    }
    events.append(event)

    snapshot = {
        "sensor_id": sensor,
        "timestamp": event["timestamp"],
        "temperature": temperature,
        "humidity": humidity,
        "battery": battery,
    }
    history[sensor].append(snapshot)
    latest[sensor] = snapshot

with open("/content/sensor_data_multi_60.json", "w", encoding="utf-8") as handle:
    json.dump(events, handle, indent=2)

print("Created 60 events")
print("Expected routes:", Counter(e["expected_route"] for e in events))
print("Scenarios:", Counter(e["scenario_type"] for e in events))

### Inspect the routing cases before calling an agent

This table is the experiment design. It lets us verify that the dataset contains enough agent cases before incurring any model cost.

In [ ]:
import pandas as pd

design = pd.DataFrame(events)[
    ["event_id", "sensor_id", "scenario_type", "expected_route",
     "temperature", "humidity", "battery", "diagnostic_note"]
]
display(design.groupby(["expected_route", "scenario_type"]).size().rename("events").reset_index())
display(design.head(12))

## 4. The simplified multi-worker workflow

The workflow remains the same:

`Read → Normalise → Precheck → (Resolved OR Agent) → Merge → Execute → Write`

The Agent PE is organised into four readable pieces:

1. `TOOLS`: the bounded functions the model may request;
2. `run_tool`: ordinary Python implementations using only the current event;
3. `run_agent`: one short model/tool loop;
4. `_process`: success result or safe human-review fallback.

The complete trace has three complementary levels:

- `routing_trace`: which precheck rule or agent route was selected;
- `agent_trace`: every agent round, including requested calls and returned evidence;
- `tool_trace`: a convenient flat list of executed evidence/action tools.

In [ ]:
%%writefile /content/sensor_workflow_multi_simple.py
from datetime import datetime, timezone
from dispel4py.base import ConsumerPE, GenericPE, IterativePE, ProducerPE
from dispel4py.workflow_graph import WorkflowGraph
from openai import OpenAI
import json
import os
import uuid

OUTPUT_FILE = "/content/multi_agent_results.jsonl"
FINAL_ACTIONS = ["accept", "retry", "maintenance", "notify_operator", "human_review"]


def now():
    return datetime.now(timezone.utc).isoformat()


def function_tool(name, description, properties=None, required=None):
    """Small helper that keeps the tool list readable."""
    return {
        "type": "function",
        "name": name,
        "description": description,
        "parameters": {
            "type": "object",
            "properties": properties or {},
            "required": required or [],
            "additionalProperties": False,
        },
        "strict": True,
    }


TOOLS = [
    function_tool("inspect_previous_readings", "Return the history snapshot attached to this event."),
    function_tool("compare_neighbours", "Compare this event with its attached neighbour snapshot."),
    function_tool(
        "request_measurement", "Create a simulated repeat-measurement request.",
        {"reason": {"type": "string"}}, ["reason"]),
    function_tool(
        "create_maintenance_ticket", "Create a simulated maintenance ticket.",
        {"reason": {"type": "string"},
         "priority": {"type": "string", "enum": ["low", "medium", "high"]}},
        ["reason", "priority"]),
    function_tool(
        "escalate_to_human", "Create a simulated human-review request.",
        {"reason": {"type": "string"}}, ["reason"]),
    function_tool(
        "submit_final_decision", "Finish with one bounded action and a reason.",
        {"action": {"type": "string", "enum": FINAL_ACTIONS},
         "reason": {"type": "string"}},
        ["action", "reason"]),
]


class ReadSensorDataPE(ProducerPE):
    def _process(self, inputs):
        with open(inputs["input"], encoding="utf-8") as handle:
            for event in json.load(handle):
                self.write("output", {
                    **event,
                    "routing_trace": [],
                    "audit": [f"{now()} read emitted {event['event_id']}"],
                })


class NormalizeDataPE(IterativePE):
    def _process(self, event):
        return {
            **event,
            "normalized_temperature": event["temperature"] / 40.0,
            "audit": event["audit"] + [f"{now()} temperature normalised"],
        }


class DeterministicPrecheckPE(GenericPE):
    """Three transparent rules; everything ambiguous goes to an agent."""
    def __init__(self):
        super().__init__()
        self._add_input("input")
        self._add_output("resolved")
        self._add_output("needs_agent")

    def _process(self, inputs):
        event = dict(inputs["input"])

        if event["battery"] < 10:
            return self.resolve(event, "battery_precheck", "maintenance",
                                f"Battery is {event['battery']}%, below 10%.")

        if not -40 <= event["temperature"] <= 85:
            return self.resolve(event, "temperature_precheck", "human_review",
                                "Temperature is outside the fixed plausible range.")

        note = event["diagnostic_note"].lower()
        if "routine reading" in note and "stable" in note:
            return self.resolve(event, "clear_normal_precheck", "accept",
                                "Plausible measurements and an explicitly stable routine note.")

        trace = event["routing_trace"] + [{
            "time": now(), "stage": "precheck", "rule": "no_rule_matched",
            "route": "agent"
        }]
        return {"needs_agent": {**event, "routing_trace": trace,
                                 "audit": event["audit"] + [f"{now()} routed to agent"]}}

    @staticmethod
    def resolve(event, rule, action, reason):
        trace = event["routing_trace"] + [{
            "time": now(), "stage": "precheck", "rule": rule,
            "route": "deterministic", "action": action
        }]
        return {"resolved": {
            **event, "action": action, "reason": reason,
            "decision_source": "deterministic_rule",
            "worker_pid": os.getpid(), "agent_trace": [], "tool_trace": [],
            "routing_trace": trace,
            "audit": event["audit"] + [f"{now()} {rule} selected {action}"],
        }}


class SimpleAgentPE(IterativePE):
    """One independent instance runs in each of four agent worker processes."""
    def __init__(self, model, max_rounds=6):
        super().__init__()
        self.model = model
        self.max_rounds = max_rounds
        self.client = None

    def preprocess(self):
        if not os.environ.get("OPENAI_API_KEY"):
            raise RuntimeError("OPENAI_API_KEY is not available to this worker.")
        self.client = OpenAI()

    def run_tool(self, name, arguments, event):
        """Execute one bounded local tool using evidence embedded in the event."""
        if name == "inspect_previous_readings":
            return {"count": len(event["previous_readings"]),
                    "readings": event["previous_readings"]}

        if name == "compare_neighbours":
            differences = [{
                "sensor_id": neighbour["sensor_id"],
                "temperature_difference": round(event["temperature"] - neighbour["temperature"], 2),
                "humidity_difference": round(event["humidity"] - neighbour["humidity"], 2),
            } for neighbour in event["neighbour_readings"]]
            return {"count": len(differences), "differences": differences,
                    "readings": event["neighbour_readings"]}

        identifiers = {
            "request_measurement": ("request_id", "RM", "repeat_measurement_requested"),
            "create_maintenance_ticket": ("ticket_id", "MT", "maintenance_ticket_created"),
            "escalate_to_human": ("review_id", "HR", "human_review_created"),
        }
        if name in identifiers:
            id_key, prefix, status = identifiers[name]
            return {id_key: f"{prefix}-{uuid.uuid4().hex[:8]}",
                    "sensor_id": event["sensor_id"], "status": status, **arguments}

        raise ValueError(f"Tool is not permitted: {name}")

    def run_agent(self, event):
        """A small model → tools → model loop with a trace of every round."""
        conversation = [{"role": "user", "content": json.dumps({
            "event_id": event["event_id"],
            "scenario_type": event["scenario_type"],
            "sensor_id": event["sensor_id"],
            "temperature": event["temperature"],
            "humidity": event["humidity"],
            "battery": event["battery"],
            "diagnostic_note": event["diagnostic_note"],
            "previous_readings_available": len(event["previous_readings"]),
            "neighbour_readings_available": len(event["neighbour_readings"]),
        }, indent=2)}]
        agent_trace, tool_trace = [], []

        instructions = """You are a bounded IoT sensor agent. Inspect previous or neighbour
evidence when it helps. For packet loss, normally request a new measurement before retry.
For likely hardware/calibration problems, normally create a maintenance ticket.
For unresolved safety ambiguity, escalate to a human. Never invent measurements.
After any needed evidence/action tools, always call submit_final_decision."""

        for round_number in range(1, self.max_rounds + 1):
            response = self.client.responses.create(
                model=self.model, instructions=instructions, input=conversation,
                tools=TOOLS, tool_choice="auto", store=False)
            calls = [item for item in response.output
                     if getattr(item, "type", None) == "function_call"]
            if not calls:
                raise RuntimeError("Agent returned no function call.")

            conversation.extend([item.model_dump(exclude_none=True)
                                 for item in response.output])
            round_trace = {"round": round_number, "calls": []}
            outputs = []

            for call in calls:
                arguments = json.loads(call.arguments)
                call_trace = {"tool": call.name, "arguments": arguments}

                if call.name == "submit_final_decision":
                    call_trace["result"] = "final_decision_accepted"
                    round_trace["calls"].append(call_trace)
                    agent_trace.append(round_trace)
                    return arguments, agent_trace, tool_trace

                result = self.run_tool(call.name, arguments, event)
                call_trace["result"] = result
                round_trace["calls"].append(call_trace)
                tool_trace.append({"round": round_number, **call_trace})
                outputs.append({"type": "function_call_output",
                                "call_id": call.call_id,
                                "output": json.dumps(result)})

            agent_trace.append(round_trace)
            conversation.extend(outputs)

        raise RuntimeError("Maximum agent rounds reached without a final decision.")

    def _process(self, event):
        if self.client is None:
            self.preprocess()
        try:
            decision, agent_trace, tool_trace = self.run_agent(event)
            return {**event, **decision, "decision_source": "openai_agent",
                    "worker_pid": os.getpid(), "agent_trace": agent_trace,
                    "tool_trace": tool_trace,
                    "audit": event["audit"] + [f"{now()} agent selected {decision['action']}"]}
        except Exception as error:
            return {**event, "action": "human_review",
                    "reason": f"Agent failure: {type(error).__name__}: {error}",
                    "decision_source": "agent_error_fallback", "worker_pid": os.getpid(),
                    "agent_trace": locals().get("agent_trace", []),
                    "tool_trace": locals().get("tool_trace", []),
                    "audit": event["audit"] + [f"{now()} safe fallback selected"]}


class MergePE(GenericPE):
    def __init__(self):
        super().__init__()
        self._add_input("deterministic")
        self._add_input("agent")
        self._add_output("output")

    def _process(self, inputs):
        value = inputs.get("deterministic", inputs.get("agent"))
        return {"output": value} if value else None


class ExecuteActionPE(IterativePE):
    def _process(self, result):
        outcomes = {
            "accept": "Reading stored as valid.",
            "retry": "Another measurement was requested.",
            "maintenance": "Maintenance workflow initiated.",
            "notify_operator": "Operator was notified.",
            "human_review": "Case sent to human review.",
        }
        return {**result, "outcome": outcomes[result["action"]],
                "audit": result["audit"] + [f"{now()} action executed"]}


class WriteResultPE(ConsumerPE):
    def _process(self, result):
        with open(OUTPUT_FILE, "a", encoding="utf-8") as handle:
            handle.write(json.dumps(result) + "\n")


read = ReadSensorDataPE(); read.name = "read"; read.numprocesses = 1
normalise = NormalizeDataPE(); normalise.numprocesses = 3
precheck = DeterministicPrecheckPE(); precheck.numprocesses = 3
agent = SimpleAgentPE(os.environ.get("OPENAI_MODEL", "gpt-5-mini")); agent.numprocesses = 4
merge = MergePE(); merge.numprocesses = 2
execute = ExecuteActionPE(); execute.numprocesses = 2
write = WriteResultPE(); write.numprocesses = 1

graph = WorkflowGraph()
graph.connect(read, "output", normalise, "input")
graph.connect(normalise, "output", precheck, "input")
graph.connect(precheck, "resolved", merge, "deterministic")
graph.connect(precheck, "needs_agent", agent, "input")
graph.connect(agent, "output", merge, "agent")
graph.connect(merge, "output", execute, "input")
graph.connect(execute, "output", write, "input")

## 5. Run with 16 worker processes and dispel4py monitoring

The process allocation is 1 reader + 3 normalisers + 3 prechecks + 4 agents + 2 mergers + 2 executors + 1 writer = 16.

In [ ]:
import os
import shutil

for path in ["/content/multi_agent_results.jsonl", "/content/results/multi_simplified"]:
    if os.path.isdir(path):
        shutil.rmtree(path)
    elif os.path.exists(path):
        os.remove(path)

!dispel4py timed_multi /content/sensor_workflow_multi_simple.py \
  -n 16 \
  -d '{"read": [{"input": "/content/sensor_data_multi_60.json"}]}' \
  --timing-dir /content/results/multi_simplified

## 6. Validate routing and summarise decisions

In [ ]:
from collections import Counter

with open("/content/multi_agent_results.jsonl", encoding="utf-8") as handle:
    results = [json.loads(line) for line in handle if line.strip()]

summary = pd.DataFrame([{
    "event_id": r["event_id"],
    "scenario_type": r["scenario_type"],
    "expected_route": r["expected_route"],
    "actual_route": "agent" if r["decision_source"] != "deterministic_rule" else "deterministic",
    "action": r["action"],
    "decision_source": r["decision_source"],
    "worker_pid": r["worker_pid"],
    "agent_rounds": len(r["agent_trace"]),
    "tool_calls": len(r["tool_trace"]),
    "tools_used": ", ".join(t["tool"] for t in r["tool_trace"]) or "none",
} for r in results])

summary["route_correct"] = summary["expected_route"] == summary["actual_route"]

print("Results:", len(summary))
print("All expected routes correct:", bool(summary["route_correct"].all()))
print("Agent-routed events:", int((summary["actual_route"] == "agent").sum()))
print("Agent worker PIDs:", sorted(summary.loc[summary.actual_route == "agent", "worker_pid"].unique()))
display(summary.groupby(["scenario_type", "actual_route", "action"]).size().rename("events").reset_index())
display(summary)

## 7. Inspect every trace level

Choose any `EVENT_ID`. The display separates routing, agent rounds, flat tool calls, and the end-to-end audit trail so the trace is readable without inspecting raw JSON.

In [ ]:
EVENT_ID = summary.loc[summary.actual_route == "agent", "event_id"].iloc[0]
selected = next(r for r in results if r["event_id"] == EVENT_ID)

print("EVENT AND FINAL DECISION")
display(pd.DataFrame([{
    "event_id": selected["event_id"],
    "scenario_type": selected["scenario_type"],
    "sensor_id": selected["sensor_id"],
    "action": selected["action"],
    "reason": selected["reason"],
    "decision_source": selected["decision_source"],
    "worker_pid": selected["worker_pid"],
}]))

print("\nROUTING TRACE")
display(pd.json_normalize(selected["routing_trace"]))

print("\nAGENT TRACE — every round, call, argument and result")
display(pd.DataFrame([
    {"round": round_item["round"], **call}
    for round_item in selected["agent_trace"]
    for call in round_item["calls"]
]))

print("\nFLAT TOOL TRACE")
display(pd.DataFrame(selected["tool_trace"]))

print("\nEND-TO-END AUDIT")
for item in selected["audit"]:
    print("•", item)

### Trace analysis across all events

In [ ]:
agent_results = [r for r in results if r["decision_source"] == "openai_agent"]
all_tools = [t["tool"] for r in agent_results for t in r["tool_trace"]]

trace_analysis = {
    "events_total": len(results),
    "deterministic_events": sum(r["decision_source"] == "deterministic_rule" for r in results),
    "agent_events": len(agent_results),
    "fallback_events": sum(r["decision_source"] == "agent_error_fallback" for r in results),
    "total_agent_rounds": sum(len(r["agent_trace"]) for r in agent_results),
    "total_executed_tools": len(all_tools),
    "tools_by_name": dict(Counter(all_tools)),
    "events_by_agent_worker": dict(Counter(str(r["worker_pid"]) for r in agent_results)),
}
display(pd.Series(trace_analysis, name="value").to_frame())

scenario_analysis = summary.groupby("scenario_type").agg(
    events=("event_id", "count"),
    agent_events=("actual_route", lambda x: (x == "agent").sum()),
    mean_agent_rounds=("agent_rounds", "mean"),
    mean_tool_calls=("tool_calls", "mean"),
).reset_index()
display(scenario_analysis)

## 8. Inspect dispel4py monitoring and workflow graphs

In [ ]:
from pathlib import Path
from IPython.display import Image, display

def newest(pattern):
    paths = list(Path("/content/results/multi_simplified").glob(pattern))
    return max(paths, key=lambda p: p.stat().st_mtime) if paths else None

monitor_files = {
    "per_pe": newest("monitor_summary_run*.csv"),
    "per_instance": newest("monitor_instances_run*.csv"),
    "per_iteration": newest("monitor_iteration_timings_run*.csv"),
    "latency_summary": newest("monitor_iteration_timings_summary_run*.csv"),
}

for name, path in monitor_files.items():
    print(f"\n{name}: {path.name if path else 'not found'}")
    if path:
        display(pd.read_csv(path))

for label, pattern in [("Abstract graph", "monitor_abstract_graph_run*.png"),
                       ("Concrete graph", "monitor_concrete_graph_run*.png")]:
    path = newest(pattern)
    if path:
        print(label)
        display(Image(filename=str(path)))

## 9. Download the complete reproducibility bundle

The ZIP contains the input, workflow source, complete JSONL trace, monitoring CSV/JSON/PNG files, and a CSV summary.

In [ ]:
import shutil
from pathlib import Path

summary.to_csv("/content/multi_agent_trace_summary.csv", index=False)

bundle = Path("/content/dispel4py_multi_agent_traces_simplified")
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir()

for filename in [
    "sensor_data_multi_60.json",
    "sensor_workflow_multi_simple.py",
    "multi_agent_results.jsonl",
    "multi_agent_trace_summary.csv",
]:
    shutil.copy2(Path("/content") / filename, bundle / filename)

shutil.copytree("/content/results/multi_simplified", bundle / "monitoring")
zip_path = shutil.make_archive(str(bundle), "zip", root_dir="/content", base_dir=bundle.name)
print("Created:", zip_path)

In [ ]:
from google.colab import files
files.download("/content/dispel4py_multi_agent_traces_simplified.zip")

## What the traces mean

- A deterministic result has a `routing_trace` and empty `agent_trace` / `tool_trace` lists.
- An agent result records the process in `worker_pid`; this shows which independent Agent PE instance handled it.
- `agent_trace` preserves the round structure and includes the final-decision call.
- `tool_trace` contains only tools actually executed, making aggregate analysis easier.
- `audit` follows the event across the workflow PEs.
- dispel4py monitoring measures workflow execution; it is different from the semantic agent/tool trace.

The workers do not exchange memory or messages. Their only shared artefact is the final output file, and a dedicated writer PE serialises those writes. Neighbour and history evidence was embedded in each event by the dataset generator.